In [ ]:
class txtFileIoOperation():
   
    def read_method1(self, file_path):
        with open(file_path, 'r') as file:
            content = file.read()
        return content

    def read_method2(self, file_path):
        a =''
        with open(file_path, 'r') as file:
            for line in file:
                a += line
        return a

    def read_method3(self, file_path):
        with open(file_path, 'r') as file:
            lines = file.readlines()

        if len(lines) >= 3:
            t_l = lines[2].strip()
            words = lines[2].split()
        else:
            return "The file has less than 3 lines."
        if len(words) >= 3:
            return words[2]

    def read_method4(self, file_path):
        with open(file_path, 'r') as file:
            lines = file.readlines()
        data = lines[2].split()[6]
        return data
   
    def write_method(self, fileName, file_path):
        data = self.read_method4(file_path)
        with open(fileName, 'w') as file:
            file.write(data)



In [ ]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("WordCount").getOrCreate()

# Path to your folder containing txt files
folder_path = "/path/to/your/folder/*.txt"

# Read all text files from the folder
text_rdd = spark.sparkContext.textFile(folder_path)

# Split lines into words and count
word_counts = (
    text_rdd.flatMap(lambda line: line.split())   # split each line into words
            .map(lambda word: (word, 1))          # map each word to (word, 1)
            .reduceByKey(lambda a, b: a + b)      # sum counts per word
)

# Collect results
for word, count in word_counts.collect():
    print(f"{word}: {count}")

# Stop Spark session
spark.stop()

##  or

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col

# Initialize Spark session
spark = SparkSession.builder.appName("WordCountDF").getOrCreate()

# Path to your folder containing txt files
folder_path = "/path/to/your/folder/*.txt"

# Read all text files into a DataFrame (each line is a row)
df = spark.read.text(folder_path)

# Split each line into words and explode into individual rows
words_df = df.select(explode(split(col("value"), "\\s+")).alias("word"))

# Group by word and count occurrences
word_counts_df = words_df.groupBy("word").count()

# Show results
word_counts_df.show(truncate=False)

# Stop Spark session
spark.stop()


In [ ]:
class GlueOp1:
    import sys
    from awsglue.transforms import *
    from awsglue.utils import getResolvedOptions
    from pyspark.context import SparkContext
    from awsglue.context import GlueContext
    from awsglue.job import Job

    # Initialize Glue context
    args = getResolvedOptions(sys.argv, ['JOB_NAME'])
    sc = SparkContext()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    job = Job(glueContext)
    job.init(args['JOB_NAME'], args)

    # Load data from S3
    datasource = glueContext.create_dynamic_frame.from_options(
        connection_type="s3",
        connection_options={"paths": ["s3://your-bucket/input-data/"]},
        format="csv",
        format_options={"withHeader": True}
    )

    # Transformation: Drop nulls and rename a column
    transformed = datasource.drop_nulls().rename_field("old_column_name", "new_column_name")

    # Write transformed data back to S3
    glueContext.write_dynamic_frame.from_options(
        frame=transformed,
        connection_type="s3",
        connection_options={"path": "s3://your-bucket/output-data/"},
        format="parquet"
    )

    job.commit()

    # Convert DynamicFrame to DataFrame
    df = datasource.toDF()
    df.show()

    # Convert Back to DynamicFrame
    dynamic_frame = DynamicFrame.fromDF(filtered_df, glueContext, "dynamic_frame")


In [ ]:
class read_s3:

    # Initialize S3 client, Define bucket and file key
    s3 = boto3.client('s3')
    bucket_name = 'your-bucket-name'
    file_key = 'path/to/your/file.txt'
    data1 = 'This is the content to upload to S3.'

    def read_s3_file():
        # Read file content
        response = s3.get_object(Bucket=bucket_name, Key=file_key)
        file_content = response['Body'].read().decode('utf-8')
        print(file_content)
   
    # Convert to DataFrame
    df = pd.read_csv(StringIO(content))

    def upload_file_s3():
        # Upload the string as a file
        s3.put_object(Bucket=bucket_name, Key=file_key, Body=data1)

    # 3. Read file from S3 into DataFrame
    df = spark.read.format("csv").option("header", "true") \
        .load("s3a://your-bucket-name/path/to/file.csv")


    spark.read.json("s3a://your-bucket/path/file.json")
    spark.read.parquet("s3a://your-bucket/path/file.parquet")


In [ ]:
class ReadTextFile:
    # read a text file and filter lines containing the word "lemon"
    from pyspark.sql import SparkSession

    # Create Spark session
    spark = SparkSession.builder.appName("FilterLemonLines").getOrCreate()
    # Read text file into RDD
    rdd = spark.sparkContext.textFile("path/to/your/textfile.txt")
    # Filter lines that contain 'lemon'
    filtered_rdd = rdd.filter(lambda line: "lemon" in line.lower())
    # Collect and print results
    for line in filtered_rdd.collect():
        print(line)
    # or
    # important Note: collect() brings all data to the driver, which can cause memory issues
    # for large datasets. For big files, prefer:
    filtered_rdd.foreach(lambda line: print(line))
    # or
    print(fltr.collect())

In [ ]:
class readCSVfile:
       # 3. Read file from S3 into DataFrame
       df = spark.read \
       .format("csv") \
       .option("header", "true") \
       .load("s3a://your-bucket-name/path/to/file.csv")


       spark.read.json("s3a://your-bucket/path/file.json")
       spark.read.parquet("s3a://your-bucket/path/file.parquet")